In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.dim_date AS
WITH date_range AS (
  SELECT DISTINCT CAST(invoice_date AS DATE) as date_value FROM automobile_repair.silver.invoice
  UNION
  SELECT DISTINCT CAST(vehicle_in_datetime AS DATE) as date_value FROM automobile_repair.silver.order
  UNION
  SELECT DISTINCT survey_sent_date as date_value FROM automobile_repair.silver.customer_survey
  UNION
  SELECT DISTINCT month as date_value FROM automobile_repair.silver.budget
)
SELECT 
  ROW_NUMBER() OVER (ORDER BY date_value) as date_key,
  date_value as full_date,
  YEAR(date_value) as year,
  MONTH(date_value) as month,
  QUARTER(date_value) as quarter,
  DAY(date_value) as day,
  DAYOFWEEK(date_value) as day_of_week,
  DAYNAME(date_value) as day_name,
  WEEKOFYEAR(date_value) as week_of_year,
  DATE_FORMAT(date_value, 'MMMM') as month_name,
  CONCAT(YEAR(date_value), '-Q', QUARTER(date_value)) as year_quarter,
  CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN 'Weekend' ELSE 'Weekday' END as day_type,
  current_timestamp() as created_at
FROM date_range
WHERE date_value IS NOT NULL
ORDER BY date_value

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.dim_store AS
SELECT 
  ROW_NUMBER() OVER (ORDER BY store_id) as store_key,
  store_id,
  store_name,
  manager_id,
  manager_name,
  current_timestamp() as created_at,
  current_timestamp() as updated_at
FROM automobile_repair.silver.store
WHERE store_id IS NOT NULL

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.dim_technician AS
SELECT DISTINCT
  ROW_NUMBER() OVER (ORDER BY technician_id) as technician_key,
  technician_id,
  technician_name,
  current_timestamp() as created_at,
  current_timestamp() as updated_at
FROM automobile_repair.silver.order
WHERE technician_id IS NOT NULL
ORDER BY technician_id

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.dim_estimator AS
SELECT DISTINCT
  ROW_NUMBER() OVER (ORDER BY estimator_id) as estimator_key,
  estimator_id,
  estimator_name,
  current_timestamp() as created_at,
  current_timestamp() as updated_at
FROM automobile_repair.silver.estimate
WHERE estimator_id IS NOT NULL
ORDER BY estimator_id

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.dim_service_type AS
SELECT DISTINCT
  ROW_NUMBER() OVER (ORDER BY service_type) as service_type_key,
  service_type,
  current_timestamp() as created_at,
  current_timestamp() as updated_at
FROM automobile_repair.silver.order
WHERE service_type IS NOT NULL
ORDER BY service_type